In [ ]:
from pythtb import Mesh, Wannier, WFArray
from pythtb.models import haldane
import numpy as np

In [ ]:
# tight-binding parameters
delta = 1
t1 = 1
t2 = -0.4
prim_model = haldane(delta, t1, t2)

print(f"Chern number: {prim_model.chern_number((0, 1), (20, 20)):0.3f}")

In [ ]:
n_super_cell = 2
model = prim_model.make_supercell([[n_super_cell, 0], [0, n_super_cell]])
model.info(show=True, short=False)

In [ ]:
nks = 20, 20  # number of k points along each dimension
mesh = Mesh(dim_k=2, axis_types=["k", "k"])
mesh.build_grid(shape=nks)
print(mesh)

In [ ]:
wfa = WFArray(model.lattice, mesh)
wfa.solve_model(model)

In [ ]:
n_orb = model.norb  # number of orbitals
n_occ = int(n_orb / 2)  # number of occupied bands (assume half-filling)

low_E_sites = np.arange(
    0, n_orb, 2
)  # low-energy sites defined to be indexed by even numbers
high_E_sites = np.arange(
    1, n_orb, 2
)  # high-energy sites defined to be indexed by odd numbers

omit_site = 6  # omitting one of the low energy sites
sites = list(np.setdiff1d(low_E_sites, [omit_site]))
tf_list = [
    [(orb, 1)] for orb in sites
]  # trial wavefunctions in form of [(orbital index, weight)]

n_tfs = len(tf_list)

print(f"Trial wavefunctions: {tf_list}")
print(f"# of Wannier functions: {n_tfs}")
print(f"# of occupied bands: {n_occ}")
print(f"Wannier fraction: {n_tfs / n_occ}")

In [ ]:
WF = Wannier(wfa)

WF.single_shot_projection(tf_list, band_idxs=list(range(n_occ)))

In [ ]:
WF.info()

In [ ]:
frozen_window = None  # frozen window in energy
outer_window = [-4, 0]  # outer window in energy

WF.disentangle(
    n_wfs=3,
    frozen_window=frozen_window,
    outer_window=outer_window,
    verbose=True,
    tf_speedup=True,
    max_iter=500,
    tol=1e-10,
)

In [ ]:
WF.info()

In [ ]:
WF.single_shot_projection(use_tilde=True)

In [ ]:
WF.info()

In [ ]:
WF.max_localize(alpha=1 / 2, max_iter=1000, tol=1e-10, grad_min=1e-10, verbose=True)

In [ ]:
WF.info()

In [ ]:
fig, ax = WF.plot_decay(0, show=True)

In [ ]:
fig, ax = WF.plot_density(0, show=True)

In [ ]:
fig, ax = WF.plot_centers(
    color_home_cell=True, center_scale=15, legend=True, pmx=4, pmy=4, show=True
)

In [ ]:
k_nodes = [
    [0, 0],
    [2 / 3, 1 / 3],
    [1 / 2, 1 / 2],
    [1 / 3, 2 / 3],
    [0, 0],
    [1 / 2, 1 / 2],
]
k_label = (r"$\Gamma $", r"$K$", r"$M$", r"$K^\prime$", r"$\Gamma $", r"$M$")

In [ ]:
n_interp = 501
interp_energies = WF.interp_bands(k_nodes, n_interp=n_interp, ret_eigvecs=False)

In [ ]:
fig, ax = model.plot_bands(
    k_nodes=k_nodes,
    nk=501,
    k_node_labels=k_label,
    proj_orb_idx=high_E_sites,
    cmap="plasma",
)

(k_vec, k_dist, k_node) = model.k_path(k_nodes, nk=n_interp, report=False)
ax.plot(k_dist, interp_energies, ls="--", c="lightgreen", lw=2, zorder=5, alpha=1)

# plot windows
if frozen_window is not None:
    ax.axhline(frozen_window[0], ls="--", c="b", label="frozen window")
    ax.axhline(frozen_window[1], ls="--", c="b")

ax.axhline(outer_window[0], ls=":", c="r", label="disentanglement window")
ax.axhline(outer_window[1], ls=":", c="r")
ax.legend()